In [1]:
import html
import pandas as pd
import ast

In [2]:
# Load data
games_raw = pd.read_csv("../data/raw/games.csv")
extended_raw = pd.read_csv("../data/raw/extended_games.csv")
train_raw = pd.read_csv("../data/raw/train_interactions.csv")
test_raw  = pd.read_csv("../data/raw/test_interactions_in.csv")

In [3]:
# Create working copies to avoid modifying the raw data
games = games_raw.copy()
extended_games = extended_raw.copy()
train = train_raw.copy()
test = test_raw.copy()

In [4]:
games.head(2)

,item_id,item_name,publisher,genres,url,tags,sentiment,metascore,specs,price,release_date
0,0,Counter-Strike,Valve,['Action'],http://store.steampowered.com/app/10/CounterSt...,"['Action', 'FPS', 'Multiplayer', 'Shooter', 'C...",Overwhelmingly Positive,88.0,"['Multi-player', 'Valve Anti-Cheat enabled']",9.99,2000-11-01
1,1,Rag Doll Kung Fu,Mark Healey,['Indie'],http://store.steampowered.com/app/1002/Rag_Dol...,"['Indie', 'Fighting', 'Multiplayer']",Mixed,69.0,"['Single-player', 'Multi-player']",9.99,2005-10-12


In [5]:
extended_games.head(2)

,item_id,item_name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,tags_Shop Keeper,tags_Coding,tags_Football (Soccer),tags_Hobby Sim,tags_Tile-Matching,tags_Mahjong,tags_Birds,tags_Football (American),tags_Fox,tags_Extraction Shooter
0,0,Counter-Strike,"Nov 1, 2000",0,9.99,0,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Rag Doll Kung Fu,"Oct 12, 2005",0,0.99,0,A piece of Steam history - THE FIRST EVER NON ...,A piece of Steam history - THE FIRST EVER NON ...,A piece of Steam history - THE FIRST EVER NON ...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
train_raw.head(2)

,user_id,item_id,item_name,playtime
0,0,0,Counter-Strike,6
1,0,2555,Day of Defeat,7


In [7]:
test_raw.head(2)

,user_id,item_id,item_name,playtime
0,4,760,Ace of Spades,83
1,4,8417,Geometry Wars: Retro Evolved,18


# games.csv

In [8]:
# Remove duplicated items and rows with missing identifiers
games = games.drop_duplicates(subset=["item_id"])
games = games.dropna(subset=["item_id", "item_name"])

In [9]:
# Columns kept 
keep_cols = [
    "item_id",
    "item_name",
    "publisher",
    "genres",
    "tags",
    "specs",
    "sentiment",
    "release_date",
]

In [10]:
games = games[keep_cols].copy()
games.head(5)

,item_id,item_name,publisher,genres,tags,specs,sentiment,release_date
0,0,Counter-Strike,Valve,['Action'],"['Action', 'FPS', 'Multiplayer', 'Shooter', 'C...","['Multi-player', 'Valve Anti-Cheat enabled']",Overwhelmingly Positive,2000-11-01
1,1,Rag Doll Kung Fu,Mark Healey,['Indie'],"['Indie', 'Fighting', 'Multiplayer']","['Single-player', 'Multi-player']",Mixed,2005-10-12
2,2,Silo 2,Nevercenter Ltd. Co.,['Animation &amp; Modeling'],"['Animation & Modeling', 'Software']",NaN,Mostly Positive,2012-12-19
3,3,Call of Duty: World at War,Activision,['Action'],"['Zombies', 'World War II', 'FPS', 'Action', '...","['Single-player', 'Multi-player', 'Co-op']",Very Positive,2008-11-18
4,4,3D-Coat V4.8,Pilgway,['Animation &amp; Modeling'],['Animation & Modeling'],['Steam Cloud'],Very Positive,2012-10-02


In [11]:
# Basic text cleaning
text_cols = ["publisher", "genres", "tags", "specs", "sentiment"]

for col in text_cols:
    games[col] = (
        games[col]
        .fillna("")
        .astype(str)
        .apply(html.unescape)
        .str.replace("\n", " ")
        .str.replace("\t", " ")
        .str.lower()
        .str.strip()
    )

games.head(5)

,item_id,item_name,publisher,genres,tags,specs,sentiment,release_date
0,0,Counter-Strike,valve,['action'],"['action', 'fps', 'multiplayer', 'shooter', 'c...","['multi-player', 'valve anti-cheat enabled']",overwhelmingly positive,2000-11-01
1,1,Rag Doll Kung Fu,mark healey,['indie'],"['indie', 'fighting', 'multiplayer']","['single-player', 'multi-player']",mixed,2005-10-12
2,2,Silo 2,nevercenter ltd. co.,['animation & modeling'],"['animation & modeling', 'software']",,mostly positive,2012-12-19
3,3,Call of Duty: World at War,activision,['action'],"['zombies', 'world war ii', 'fps', 'action', '...","['single-player', 'multi-player', 'co-op']",very positive,2008-11-18
4,4,3D-Coat V4.8,pilgway,['animation & modeling'],['animation & modeling'],['steam cloud'],very positive,2012-10-02


In [12]:
games["publisher"] = games["publisher"].replace("", "unknown")
games["release_date"] = pd.to_datetime(games["release_date"], errors="coerce")

In [13]:
games.head(5)

,item_id,item_name,publisher,genres,tags,specs,sentiment,release_date
0,0,Counter-Strike,valve,['action'],"['action', 'fps', 'multiplayer', 'shooter', 'c...","['multi-player', 'valve anti-cheat enabled']",overwhelmingly positive,2000-11-01
1,1,Rag Doll Kung Fu,mark healey,['indie'],"['indie', 'fighting', 'multiplayer']","['single-player', 'multi-player']",mixed,2005-10-12
2,2,Silo 2,nevercenter ltd. co.,['animation & modeling'],"['animation & modeling', 'software']",,mostly positive,2012-12-19
3,3,Call of Duty: World at War,activision,['action'],"['zombies', 'world war ii', 'fps', 'action', '...","['single-player', 'multi-player', 'co-op']",very positive,2008-11-18
4,4,3D-Coat V4.8,pilgway,['animation & modeling'],['animation & modeling'],['steam cloud'],very positive,2012-10-02


In [14]:
def clean_list_str(x):
    """Clean text values."""
    if not isinstance(x, str):
        return ""
    x = x.strip()
    if x == "":
        return ""
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return " ".join(str(t).lower().strip() for t in parsed)
    except:
        return x.lower().strip()

    return x.lower().strip()

# Columns that may contain multiple values in one field
list_cols = ["genres", "tags", "specs"]

for col in list_cols:
    games[col] = games[col].apply(clean_list_str)

In [15]:
games.dtypes

item_id                  int64
item_name               object
publisher               object
genres                  object
tags                    object
specs                   object
sentiment               object
release_date    datetime64[ns]
dtype: object

In [16]:
games.head(5)

,item_id,item_name,publisher,genres,tags,specs,sentiment,release_date
0,0,Counter-Strike,valve,action,action fps multiplayer shooter classic team-ba...,multi-player valve anti-cheat enabled,overwhelmingly positive,2000-11-01
1,1,Rag Doll Kung Fu,mark healey,indie,indie fighting multiplayer,single-player multi-player,mixed,2005-10-12
2,2,Silo 2,nevercenter ltd. co.,animation & modeling,animation & modeling software,,mostly positive,2012-12-19
3,3,Call of Duty: World at War,activision,action,zombies world war ii fps action multiplayer sh...,single-player multi-player co-op,very positive,2008-11-18
4,4,3D-Coat V4.8,pilgway,animation & modeling,animation & modeling,steam cloud,very positive,2012-10-02


In [17]:
games.to_csv("../data/cleansed/games_cleansed.csv", index=False)

# extended_games.csv

In [18]:
extended_games = extended_games.drop_duplicates(subset=["item_id"])
extended_games = extended_games.dropna(subset=["item_id"])

In [19]:
# Useful columns from the extended dataset
keep_cols_ext = [
    "item_id",
    "item_name",
    "release_date",
    "required_age",
    "price",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "peak_ccu",
    "windows",
    "mac",
    "linux"
]
extended_games_cleansed = extended_games[keep_cols_ext].copy()

In [20]:
extended_games_cleansed["release_date"] = pd.to_datetime(extended_games_cleansed["release_date"], errors="coerce")

In [21]:
# Convert numeric columns to the correct type
numeric_cols_ext = [
    "required_age",
    "price",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "peak_ccu",
]
for col in numeric_cols_ext:
    extended_games_cleansed[col] = pd.to_numeric(extended_games_cleansed[col], errors="coerce")

In [22]:
# Normalize OS support columns to boolean values
for col in ["windows", "mac", "linux"]:
    extended_games_cleansed[col] = (
        extended_games_cleansed[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )

In [23]:
extended_games_cleansed.head(10)
extended_games_cleansed.dtypes

item_id                              int64
item_name                           object
release_date                datetime64[ns]
required_age                         int64
price                              float64
positive                             int64
negative                             int64
recommendations                      int64
average_playtime_forever             int64
peak_ccu                             int64
windows                               bool
mac                                   bool
linux                                 bool
dtype: object

In [24]:
extended_games_cleansed.to_csv("../data/cleansed/extended_games_cleansed.csv", index=False)

# extended_games_extra_tags.csv

In [25]:
ext_tag_cols = [c for c in extended_games.columns if c.startswith("tags_")]

extended_games[ext_tag_cols] = extended_games[ext_tag_cols].apply(
    pd.to_numeric, errors="coerce"
)

In [26]:
games_tags_set = set()

def collect_tags(cell):
    """Collect unique tags from the tags column."""
    if not isinstance(cell, str) or cell.strip() == "":
        return
    try:
        tags = ast.literal_eval(cell)
    except:
        return
    for t in tags:
        if isinstance(t, str):
            games_tags_set.add(t.strip().lower())

games["tags"].apply(collect_tags)

print("Games cleansed:", len(games_tags_set))

Games cleansed: 0


In [27]:
ext_tag_cols = [c for c in extended_games.columns if c.startswith("tags_")]
print("Total columns (ext_tag_cols):", len(ext_tag_cols))
print(ext_tag_cols[:10])

Total columns (ext_tag_cols): 448
['tags_Indie', 'tags_Casual', 'tags_Sports', 'tags_Bowling', 'tags_Action', 'tags_Pixel Graphics', 'tags_2D', 'tags_Retro', 'tags_Arcade', 'tags_Score Attack']


In [28]:
ext_tag_names = {
    col: col.replace("tags_", "").strip().lower()
    for col in ext_tag_cols
}

In [29]:
def is_good_extra_tag(t):
    """Check if a tag is usable."""
    if t in games_tags_set:
        return False
    if len(t) < 3:
        return False
    if any(ch.isdigit() for ch in t):
        return False
    return True

extra_tag_cols = {
    col: tagname
    for col, tagname in ext_tag_names.items()
    if is_good_extra_tag(tagname)
}

print("Tags:", len(extra_tag_cols))
print(list(extra_tag_cols.values())[:20])

Tags: 430
['indie', 'casual', 'sports', 'bowling', 'action', 'pixel graphics', 'retro', 'arcade', 'score attack', 'minimalist', 'comedy', 'singleplayer', 'fast-paced', 'funny', 'parody', 'difficult', 'gore', 'violent', 'western', 'controller']


In [30]:
def extract_extra_tags(row):
    """Extract extra tags from extended features."""
    tags = []
    for col, tagname in extra_tag_cols.items():
        val = row.get(col)
        if pd.isna(val):
            continue
        if float(val) > 0:
            tags.append(tagname)
    return tags


extended_games["ext_tags"] = extended_games.apply(extract_extra_tags, axis=1)

/var/folders/3y/_fxcpryx5f9f989qc94cmzwh0000gn/T/ipykernel_70545/3106427992.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  extended_games["ext_tags"] = extended_games.apply(extract_extra_tags, axis=1)


In [31]:
extended_extra_tags = extended_games[["item_id", "item_name", "ext_tags"]].copy()


extended_extra_tags.to_csv(
    "../data/cleansed/extended_games_extra_tags.csv",
    index=False
)

extended_extra_tags.head(10)

,item_id,item_name,ext_tags
0,0,Counter-Strike,"[action, score attack, survival, multiplayer, ..."
1,1,Rag Doll Kung Fu,"[indie, action, singleplayer, physics, multipl..."
2,2,Silo 2,"[software, animation & modeling]"
3,3,Call of Duty: World at War,"[action, singleplayer, gore, survival, adventu..."
4,13,Runespell: Overture,"[indie, casual, singleplayer, adventure, rpg, ..."
5,14,Dead Mountaineer's Hotel,"[adventure, point & click]"
6,16,Vertex Dispenser,"[indie, action, strategy]"
7,17,PT Boats: Knights of the Sea,"[simulation, naval]"
8,19,PT Boats: South Gambit,"[simulation, world war ii, naval]"
9,20,Orcs Must Die!,"[indie, action, arcade, score attack, comedy, ..."


# train_interactions.csv

In [32]:
train = train.drop_duplicates()

In [33]:
# Ensure numeric types for training interactions
train["user_id"] = pd.to_numeric(train["user_id"], errors="coerce")
train["item_id"] = pd.to_numeric(train["item_id"], errors="coerce")
train["playtime"] = pd.to_numeric(train["playtime"], errors="coerce")

In [34]:
train = train.dropna(subset=["user_id", "item_id", "playtime"])

In [35]:
# Cast identifiers to int after cleaning
train["user_id"] = train["user_id"].astype(int)
train["item_id"] = train["item_id"].astype(int)

In [36]:
train = train[train["playtime"] > 0]

In [37]:
train.to_csv("../data/cleansed/train_interactions_cleansed.csv", index=False)